# 00b. Consolidate Vectors & Fix Sixpack_Rips Relative

**두 가지 기능:**
1. Google Drive 내 산발적 벡터 파일을 `Final_Vector/`로 통합
2. Sixpack_Rips의 `relative` barcode를 올바른 H*(K,L)로 재계산하여 대체

## 1. 환경 설정

In [ ]:
import os, glob, shutil, time, gc
import numpy as np
import pandas as pd
import psutil

from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/URP'
TOTAL_SIMS = 512

SOURCE_DIRS = {
    'Inter_PI':       [os.path.join(BASE_DIR, '1224_Vectors', 'Inter_PI')],
    '3D_PI':          [os.path.join(BASE_DIR, '1224_Vectors', '3D_PI')],
    'Ord_PI':         [os.path.join(BASE_DIR, '1224_Vectors', 'Ord_PI')],
    'Sixpack_Rips':   [os.path.join(BASE_DIR, '1224_Vectors', 'Sixpack_Rips')],
    'Sixpack_Chroma': [os.path.join(BASE_DIR, '1224_Vectors', 'Sixpack_Chroma')],
}

OUTPUT_BASE = os.path.join(BASE_DIR, 'Final_Vector')
os.makedirs(OUTPUT_BASE, exist_ok=True)
print(f'Output: {OUTPUT_BASE}')

## 2. 소스 스캔 & 파일 복사

In [ ]:
def scan_sources(source_dirs, descriptor):
    files = {}
    for src_dir in source_dirs:
        if not os.path.exists(src_dir): continue
        for fp in glob.glob(os.path.join(src_dir, f'{descriptor}_*.npz')):
            try:
                sim_idx = int(os.path.basename(fp).split('_')[-1].split('.')[0])
                if sim_idx not in files or os.path.getmtime(fp) > os.path.getmtime(files[sim_idx]):
                    files[sim_idx] = fp
            except ValueError: pass
    return files

print('=== 소스 스캔 ===')
all_scanned = {}
for desc, dirs in SOURCE_DIRS.items():
    files = scan_sources(dirs, desc)
    all_scanned[desc] = files
    missing = TOTAL_SIMS - len(files)
    status = '✓' if missing == 0 else f'✗ ({missing} missing)'
    print(f'  {desc:<20s}: {len(files):>4d}/{TOTAL_SIMS}  {status}')

In [ ]:
for desc in SOURCE_DIRS:
    dest_dir = os.path.join(OUTPUT_BASE, desc)
    os.makedirs(dest_dir, exist_ok=True)
    files = all_scanned[desc]
    copied, skipped = 0, 0
    for sim_idx in sorted(files.keys()):
        src = files[sim_idx]
        dest = os.path.join(dest_dir, f'{desc}_{sim_idx}.npz')
        if os.path.exists(dest) and os.path.getsize(dest) == os.path.getsize(src):
            skipped += 1; continue
        shutil.copy2(src, dest); copied += 1
    print(f'{desc:<20s}: copied={copied}, skipped={skipped}')

---
## 3. Sixpack_Rips의 `relative` barcode 재계산

기존 Sixpack_Rips npz 파일 내 `relative` PI를,
L=Rips(A) ⊆ K=Rips(A∪B) inclusion의 올바른 **H*(K,L)** barcode PI로 대체합니다.

참고: `Phase/Phase8_Relative_Rips.ipynb`

In [ ]:
from gudhi import RipsComplex
from persim import PersistenceImager
import persim.images_weights as weights
from collections import defaultdict

In [ ]:
def compute_Rips(points, max_edge=10):
    rips = RipsComplex(points=points, max_edge_length=max_edge)
    return rips.create_simplex_tree(max_dimension=2)

def divide_filtration(st):
    pairs = [(tuple(sorted(s)), f) for s, f in st.get_filtration()]
    return [p[0] for p in pairs], [p[1] for p in pairs]

def compute_relative_barcode(A, B, max_edge=10):
    """H*(K, L): L=Rips(A) ⊆ K=Rips(A∪B)의 Relative Persistence Barcode."""
    total = np.concatenate([A, B], axis=0)
    a_len = len(A)
    st = compute_Rips(total, max_edge=max_edge)
    simplices, filt = divide_filtration(st)
    del st, total; gc.collect()

    idx_KmL = [i for i, s in enumerate(simplices) if any(v >= a_len for v in s)]
    set_idx_KmL = set(idx_KmL)
    KmL_pos = {g: l for l, g in enumerate(idx_KmL)}

    sf_to_idx = {s: i for i, s in enumerate(simplices)}
    Drel = []
    for g_idx in idx_KmL:
        s = simplices[g_idx]; rows = set()
        if len(s) > 1:
            for j in range(len(s)):
                face = s[:j] + s[j+1:]
                fi = sf_to_idx.get(face)
                if fi in set_idx_KmL: rows.add(KmL_pos[fi])
        Drel.append(rows)
    del sf_to_idx, set_idx_KmL, KmL_pos; gc.collect()

    # Column reduction
    m = len(Drel); R = [set(col) for col in Drel]
    low = [-1]*m; pivot = {}
    for i in range(m):
        while R[i]:
            li = max(R[i])
            if li in pivot: R[i] ^= R[pivot[li]]
            else: pivot[li] = i; low[i] = li; break
    del Drel; gc.collect()

    rel_bars = defaultdict(list)
    for pos in range(len(idx_KmL)):
        if low[pos] != -1:
            sigma = idx_KmL[low[pos]]; tau = idx_KmL[pos]
            b, d = filt[sigma], filt[tau]
            if abs(b-d) > 1e-12:
                rel_bars[len(simplices[sigma])-1].append((b, d))
    del R, low, idx_KmL, simplices, filt; gc.collect()

    out = {}
    for p in [0, 1]:
        if p in rel_bars and rel_bars[p]:
            arr = np.array(rel_bars[p])
            out[p] = arr[np.lexsort((arr[:,1], arr[:,0]))]
        else: out[p] = np.empty((0, 2))
    return out

def compute_PIs_relative(barcodes, max_eps=20, px_res=0.1, sigma=0.05):
    """Relative barcode → PI vector."""
    vector = {}
    for key in barcodes:
        if len(barcodes[key]) == 0: barcodes[key] = np.zeros((0,2))
    # H0
    pi0 = PersistenceImager(pixel_size=px_res, birth_range=(0,1), pers_range=(0,max_eps))
    pi0.weight = weights.persistence; pi0.weight_params = {'n':1}
    pi0.kernel_params = {'sigma': [[sigma,0],[0,sigma]]}
    bars0 = np.array(barcodes.get(0, np.zeros((0,2))))
    img0 = pi0.transform(bars0, skew=False) if len(bars0)>0 else np.zeros((int(1/px_res), int(max_eps/px_res)))
    vector[0] = np.mean(img0, axis=0)
    # H1
    pi1 = PersistenceImager(pixel_size=px_res, birth_range=(0,max_eps), pers_range=(0,max_eps/2))
    pi1.weight = weights.persistence; pi1.weight_params = {'n':1}
    pi1.kernel_params = {'sigma': [[sigma,0],[0,sigma]]}
    bars1 = np.array(barcodes.get(1, np.zeros((0,2))))
    img1 = pi1.transform(bars1, skew=True) if len(bars1)>0 else np.zeros((int(max_eps/px_res), int((max_eps/2)/px_res)))
    vector[1] = img1.flatten()
    return vector

print('Relative barcode functions defined.')

### 3-1. 재계산 실행
시뮬레이션 원본(Pos/Types) → Relative H*(K,L) barcode → PI →
기존 Sixpack_Rips npz의 `relative` 키를 대체하여 저장.

In [ ]:
MAX_EDGE = 10
A_VALS = [0.0, 0.01, 0.05, 0.09, 0.13, 0.17, 0.21, 0.25]
PARAM_LIST = [(x1, x2, x3) for x1 in A_VALS for x2 in A_VALS for x3 in A_VALS]

RIPS_DIR = os.path.join(OUTPUT_BASE, 'Sixpack_Rips')

def get_ram_mb():
    return psutil.Process(os.getpid()).memory_info().rss / 1024 / 1024

START_IDX = 1
END_IDX = 512
updated, skipped, errors = 0, 0, 0

for idx in range(START_IDX, END_IDX + 1):
    rips_path = os.path.join(RIPS_DIR, f'Sixpack_Rips_{idx}.npz')
    if not os.path.exists(rips_path):
        skipped += 1; continue

    # 시뮬레이션 원본 파일
    params = PARAM_LIST[idx - 1]
    folder = os.path.join(BASE_DIR, f'ParamSweep_{idx}_Output')
    pos_file = os.path.join(folder, f'Pos_{params[0]:.2f}_{params[1]:.2f}_{params[2]:.2f}.dat')
    types_file = os.path.join(folder, f'Types_{params[0]:.2f}_{params[1]:.2f}_{params[2]:.2f}.dat')

    if not os.path.exists(pos_file):
        skipped += 1; continue

    print(f'[{idx:>3d}/512] Relative 재계산 (RAM: {get_ram_mb():.0f}MB)', end=' ')
    t0 = time.time()

    try:
        types = np.loadtxt(types_file, dtype=int)
        positions = np.loadtxt(pos_file, delimiter=',')
        A = positions[types == 1]; B = positions[types == 2]
        del types, positions

        # Relative barcode → PI
        rel_A2B = compute_relative_barcode(A, B, max_edge=MAX_EDGE)
        rel_B2A = compute_relative_barcode(B, A, max_edge=MAX_EDGE)
        del A, B; gc.collect()

        PI_rel_A2B = compute_PIs_relative(rel_A2B, max_eps=MAX_EDGE*2)
        PI_rel_B2A = compute_PIs_relative(rel_B2A, max_eps=MAX_EDGE*2)
        del rel_A2B, rel_B2A; gc.collect()

        # 기존 Sixpack_Rips 로드 → relative 교체
        data = np.load(rips_path, allow_pickle=True)
        sp_A2B = data['arr_0'].item()  # dict
        sp_B2A = data['arr_1'].item()  # dict

        sp_A2B['relative'] = PI_rel_A2B
        sp_B2A['relative'] = PI_rel_B2A

        # 덮어쓰기 저장
        np.savez_compressed(rips_path, sp_A2B, sp_B2A)
        del data, sp_A2B, sp_B2A, PI_rel_A2B, PI_rel_B2A; gc.collect()

        updated += 1
        print(f'OK ({time.time()-t0:.1f}s)')

    except Exception as e:
        errors += 1
        print(f'ERROR: {e}')

print(f'\n=== 완료 ===')
print(f'Updated: {updated}, Skipped: {skipped}, Errors: {errors}')

## 4. 최종 검증

In [ ]:
print('=== Final_Vector 최종 구조 ===')
total_size = 0
for desc in SOURCE_DIRS:
    d = os.path.join(OUTPUT_BASE, desc)
    files = glob.glob(os.path.join(d, '*.npz'))
    size = sum(os.path.getsize(f) for f in files)
    total_size += size
    missing = TOTAL_SIMS - len(files)
    status = '✓' if missing == 0 else f'✗ ({missing} missing)'
    print(f'  {desc:<20s}: {len(files):>4d} files, {size/1024/1024:.1f}MB  {status}')
print(f'  Total: {total_size/1024/1024:.1f}MB')

# Sixpack_Rips relative 교체 확인
test_file = os.path.join(OUTPUT_BASE, 'Sixpack_Rips', 'Sixpack_Rips_1.npz')
if os.path.exists(test_file):
    d = np.load(test_file, allow_pickle=True)
    sp = d['arr_0'].item()
    print(f'\nSixpack_Rips_1 keys: {list(sp.keys())}')
    if 'relative' in sp:
        rel = sp['relative']
        if isinstance(rel, dict):
            print(f'  relative: dict with keys {list(rel.keys())}')
            for k in rel:
                v = rel[k]
                if hasattr(v, 'shape'): print(f'    [{k}]: shape={v.shape}')
                else: print(f'    [{k}]: type={type(v)}')
        else:
            print(f'  relative: type={type(rel)}')
    print('\n✓ relative 키가 재계산된 H*(K,L) PI로 대체되었습니다.')

## (선택) Drive 전체 검색

In [ ]:
# 누락된 벡터 파일을 Drive 전체에서 검색 (필요시 실행)
# SEARCH_DESC = 'Sixpack_Rips'
# for dirpath, _, filenames in os.walk('/content/drive/MyDrive'):
#     depth = dirpath.replace('/content/drive/MyDrive','').count(os.sep)
#     if depth > 4: continue
#     for fn in filenames:
#         if fn.startswith(f'{SEARCH_DESC}_') and fn.endswith('.npz'):
#             print(os.path.join(dirpath, fn))